# Test Pseudo-Ground-Truth Workbench + Evaluation

이 노트북은 기존 학습 노트북에 붙이지 않고 따로 쓰는 검수/평가용 노트북입니다.

목적은 세 가지입니다.

1. `pill_class_reference_grid.png`와 `pill_class_reference_table.csv`로 class id ↔ 약 이름 ↔ 대표 이미지를 붙입니다.
2. `submission.csv`의 test 예측을 사람이 고칠 수 있는 `test_pseudo_ground_truth_review.csv`로 변환합니다.
3. 검수된 pseudo-ground-truth가 있으면 mAP/confusion matrix를 계산합니다. 기존 `val.json`/`val_predictions.json`으로 validation 평가는 바로 계산합니다.

주의: Kaggle/대회 서버의 숨겨진 test 정답은 이 파일들만으로 자동 복원할 수 없습니다. 여기서 만드는 test 정답은 **이미지 보드와 매핑 CSV를 기준으로 사람이 검수해서 만드는 pseudo/manual ground truth**입니다.


In [ ]:
# =========================
# 0. Paths & Parameters
# =========================
from pathlib import Path
import os
import sys
import json
import math
import shutil
import warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd

from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display, Image as IPyImage, Markdown

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:
    sns = None

try:
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
except Exception as exc:
    COCO = None
    COCOeval = None
    print("pycocotools import failed:", repr(exc))


def find_project_dir(start: Path | None = None):
    start = Path(start or Path.cwd()).resolve()
    markers = [
        "working/reports/pill_class_reference_table.csv",
        "working/reports/pill_class_reference_grid.png",
        "sprint_ai_project1_data/test_images",
    ]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError(f"Could not find project root from {start}")

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "sprint_ai_project1_data"
TEST_IMG_DIR = DATA_DIR / "test_images"

REF_GRID = PROJECT_DIR / "working/reports/pill_class_reference_grid.png"
REF_TABLE = PROJECT_DIR / "working/reports/pill_class_reference_table.csv"
CLASS_NUMBER_MAP = PROJECT_DIR / "working/reports/pill_class_number_map.csv"
SUBMISSION_CSV = PROJECT_DIR / "working/submission.csv"

COCO_DIR = PROJECT_DIR / "working/pill_coco"
VAL_GT_JSON = COCO_DIR / "annotations/val.json"
VAL_PREDS_JSON = COCO_DIR / "val_predictions.json"
IDX2CAT_JSON = COCO_DIR / "idx2cat.json"

OUT_DIR = PROJECT_DIR / "working/reports/test_pseudo_gt_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMISSION_NAMED_CSV = OUT_DIR / "submission_with_drug_names.csv"
PSEUDO_SEED_CSV = OUT_DIR / "test_pseudo_ground_truth_seed_from_submission.csv"
PSEUDO_REVIEW_CSV = OUT_DIR / "test_pseudo_ground_truth_review.csv"
PSEUDO_GT_CSV = OUT_DIR / "test_pseudo_ground_truth.csv"
PSEUDO_GT_COCO_JSON = OUT_DIR / "test_pseudo_ground_truth_coco.json"

SCORE_THR_SEED = 0.001
MAX_DETS_PER_IMAGE = 4
IOU_THRESHOLD = 0.75

print("PROJECT_DIR:", PROJECT_DIR)
print("OUT_DIR:", OUT_DIR)


## 1. Input Check

아래 셀은 필요한 파일이 있는지 확인합니다. `pill_class_reference_table.csv`는 약 이름표이고, `submission.csv`는 test 예측 후보입니다.


In [ ]:
required_paths = {
    "class reference grid": REF_GRID,
    "class reference table": REF_TABLE,
    "class number map": CLASS_NUMBER_MAP,
    "submission": SUBMISSION_CSV,
    "test images": TEST_IMG_DIR,
    "validation ground truth": VAL_GT_JSON,
    "validation predictions": VAL_PREDS_JSON,
    "idx2cat": IDX2CAT_JSON,
}

for name, path in required_paths.items():
    print(f"{name:28s}", path.exists(), path)

missing = [name for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")


## 2. Load Reference Board/Table

이 표가 `category_id → 약 이름/영문명/대표 crop` 매핑입니다. test 이미지의 숨겨진 정답을 서버에서 가져오는 것이 아니라, 이 보드를 기준으로 사람이 비교/검수할 수 있게 만듭니다.


In [ ]:
ref = pd.read_csv(REF_TABLE)
ref["category_id"] = ref["category_id"].astype(int)

if CLASS_NUMBER_MAP.exists():
    number_map = pd.read_csv(CLASS_NUMBER_MAP)
    number_map["category_id"] = number_map["category_id"].astype(int)
    number_map["class_no"] = number_map["class_no"].astype(int)
else:
    number_map = ref[["category_id"]].sort_values("category_id").reset_index(drop=True)
    number_map.insert(0, "class_no", np.arange(1, len(number_map) + 1))
    number_map.to_csv(CLASS_NUMBER_MAP, index=False, encoding="utf-8-sig")

if "class_no" in ref.columns:
    ref = ref.drop(columns=["class_no"])
ref = ref.merge(number_map[["class_no", "category_id"]], on="category_id", how="left")
ref["class_no"] = ref["class_no"].astype(int)
ref["n_number"] = ref["class_no"].map(lambda x: f"N{int(x):02d}")
ref = ref.sort_values("class_no").reset_index(drop=True)

category_to_n_number = ref.set_index("category_id")["n_number"].to_dict()
n_number_to_category_id = ref.set_index("n_number")["category_id"].to_dict()
category_to_name = ref.set_index("category_id")["name"].astype(str).to_dict()

def normalize_n_number(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().upper()
    if not text or text in {"NAN", "NONE", "NULL"}:
        return ""
    if text.startswith("N"):
        text = text[1:]
    if not text.isdigit():
        raise ValueError(f"Invalid N-number: {value!r}. Use forms like N01 or 1.")
    return f"N{int(text):02d}"


def category_id_from_n_number(value):
    n_number = normalize_n_number(value)
    if not n_number:
        return None
    if n_number not in n_number_to_category_id:
        raise ValueError(f"Unknown N-number: {value!r}. Valid range is N01-N{len(ref):02d}.")
    return int(n_number_to_category_id[n_number])

ref_cols = [
    "class_no", "n_number", "category_id", "name", "name_en", "company", "shape", "color",
    "print_front", "print_back", "count", "sample_image_id", "sample_file_name", "sample_bbox", "sample_path"
]
display(ref[ref_cols].head(12))

print("classes:", len(ref))
print("N-number range:", f"N01-N{len(ref):02d}")
display(IPyImage(filename=str(REF_GRID)))


## 3. Convert Submission Into a Reviewable Pseudo-GT Seed

`submission.csv`의 예측을 그대로 정답이라고 부르면 안 됩니다. 대신 이 셀은 예측 결과에 약 이름과 N-number를 붙여 **수동 검수용 seed 파일**을 만듭니다.

검수 방법:

- `predicted_n_number`: 모델이 분류한 번호입니다. 직접 수정하지 않는 것을 권장합니다.
- `correct_n_number`: 틀렸을 때만 `N07`처럼 정답 번호를 적습니다. 비워두면 `predicted_n_number`를 그대로 사용합니다.
- `keep`: 이 박스를 정답으로 쓸 거면 `1`, 버릴 거면 `0`
- `bbox_x`, `bbox_y`, `bbox_w`, `bbox_h`: 박스가 틀렸으면 수정
- 누락된 약은 새 row로 추가하고 `correct_n_number`, bbox를 채웁니다.
- 중복 박스는 하나만 `keep=1`
- 검수한 row는 `review_status=reviewed`로 바꾸면 나중에 필터링하기 쉽습니다.


In [ ]:
submission = pd.read_csv(SUBMISSION_CSV)
submission["category_id"] = submission["category_id"].astype(int)

known_categories = set(ref["category_id"].astype(int))
unknown_categories = sorted(set(submission["category_id"]) - known_categories)
if unknown_categories:
    raise ValueError(f"Submission contains unknown category_id values: {unknown_categories[:20]}")

label_cols = [
    "category_id", "class_no", "n_number", "name", "name_en", "company", "shape", "color",
    "print_front", "print_back", "sample_path", "sample_bbox",
]
sub_named = submission.merge(ref[label_cols], on="category_id", how="left")
sub_named = sub_named.sort_values(["image_id", "score"], ascending=[True, False]).reset_index(drop=True)
sub_named["candidate_rank"] = sub_named.groupby("image_id").cumcount() + 1
sub_named = sub_named.rename(columns={"n_number": "predicted_n_number", "name": "predicted_drug_name"})
sub_named["correct_n_number"] = ""
sub_named.to_csv(SUBMISSION_NAMED_CSV, index=False, encoding="utf-8-sig")

pseudo_seed = sub_named.copy()
pseudo_seed.insert(0, "keep", (pseudo_seed["score"] >= SCORE_THR_SEED).astype(int))
pseudo_seed.insert(1, "review_status", "unreviewed")
pseudo_seed.insert(2, "review_note", "")
pseudo_seed.insert(3, "source", "submission_seed")
pseudo_seed = pseudo_seed[
    [
        "keep", "review_status", "review_note", "source", "annotation_id", "image_id", "candidate_rank",
        "predicted_n_number", "correct_n_number", "category_id", "predicted_drug_name", "name_en",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score",
        "company", "shape", "color", "print_front", "print_back", "sample_path", "sample_bbox",
    ]
]
pseudo_seed.to_csv(PSEUDO_SEED_CSV, index=False, encoding="utf-8-sig")

if not PSEUDO_REVIEW_CSV.exists():
    pseudo_seed.to_csv(PSEUDO_REVIEW_CSV, index=False, encoding="utf-8-sig")
    print("created review CSV:", PSEUDO_REVIEW_CSV)
else:
    review_existing = pd.read_csv(PSEUDO_REVIEW_CSV)
    if "predicted_n_number" not in review_existing.columns or "correct_n_number" not in review_existing.columns:
        key_cols = ["annotation_id", "predicted_n_number", "correct_n_number"]
        n_seed = pseudo_seed[key_cols].copy()
        if "correct_n_number" in review_existing.columns:
            n_seed = n_seed.drop(columns=["correct_n_number"])
        review_existing = review_existing.merge(n_seed, on="annotation_id", how="left")
        if "correct_n_number" not in review_existing.columns:
            review_existing.insert(review_existing.columns.get_loc("category_id"), "correct_n_number", "")
        if "predicted_n_number" not in review_existing.columns:
            review_existing.insert(review_existing.columns.get_loc("correct_n_number"), "predicted_n_number", review_existing["category_id"].map(category_to_n_number))
        review_existing.to_csv(PSEUDO_REVIEW_CSV, index=False, encoding="utf-8-sig")
        print("patched existing review CSV:", PSEUDO_REVIEW_CSV)
    else:
        print("review CSV already exists with N-number columns, not overwritten:", PSEUDO_REVIEW_CSV)

print("submission rows:", len(submission))
print("test images in submission:", submission["image_id"].nunique())
print("named submission:", SUBMISSION_NAMED_CSV)
print("seed pseudo-GT:", PSEUDO_SEED_CSV)

display(pseudo_seed.head(12))


## 4. Visual Review Contact Sheets

아래 함수들은 test 이미지 위에 후보 박스를 그리고, 오른쪽에 예측 class의 reference crop/name을 붙입니다. 이걸 보고 `test_pseudo_ground_truth_review.csv`를 고치면 됩니다.


In [ ]:
FONT_CANDIDATES = [
    "/System/Library/Fonts/AppleSDGothicNeo.ttc",
    "/Library/Fonts/AppleGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
]
FONT_PATH = next((Path(p) for p in FONT_CANDIDATES if Path(p).exists()), None)


def get_font(size=16):
    if FONT_PATH is None:
        return ImageFont.load_default()
    try:
        return ImageFont.truetype(str(FONT_PATH), size=size, index=0)
    except TypeError:
        return ImageFont.truetype(str(FONT_PATH), size=size)

FONT_LABEL = get_font(18)
FONT_SMALL = get_font(13)
FONT_TINY = get_font(11)


def parse_bbox_string(value):
    if pd.isna(value):
        return None
    parts = [float(x) for x in str(value).replace("[", "").replace("]", "").split(",")]
    if len(parts) != 4:
        return None
    return parts


def crop_xywh(image: Image.Image, bbox, pad_ratio=0.25):
    x, y, w, h = [float(v) for v in bbox]
    W, H = image.size
    pad = max(w, h) * pad_ratio
    x1 = max(0, int(round(x - pad)))
    y1 = max(0, int(round(y - pad)))
    x2 = min(W, int(round(x + w + pad)))
    y2 = min(H, int(round(y + h + pad)))
    return image.crop((x1, y1, x2, y2)).convert("RGB")


def reference_crop(category_id, size=(110, 110)):
    row = ref[ref["category_id"] == int(category_id)]
    if row.empty:
        return Image.new("RGB", size, "#f3f4f6")
    row = row.iloc[0]
    path = Path(str(row["sample_path"]))
    bbox = parse_bbox_string(row["sample_bbox"])
    if not path.exists() or bbox is None:
        return Image.new("RGB", size, "#f3f4f6")
    crop = crop_xywh(Image.open(path).convert("RGB"), bbox)
    thumb = ImageOps.contain(crop, size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", size, "#f3f4f6")
    canvas.paste(thumb, ((size[0] - thumb.width) // 2, (size[1] - thumb.height) // 2))
    return canvas

def row_drug_name(row):
    value = row.get("predicted_drug_name", row.get("name", ""))
    if pd.isna(value):
        return ""
    return str(value)



def draw_test_review_image(image_id, review_df=None, score_thr=0.0, width=760):
    if review_df is None:
        review_df = pd.read_csv(PSEUDO_REVIEW_CSV)
    image_id = int(image_id)
    image_path = TEST_IMG_DIR / f"{image_id}.png"
    if not image_path.exists():
        raise FileNotFoundError(image_path)

    rows = review_df[(review_df["image_id"].astype(int) == image_id) & (review_df["score"].fillna(1) >= score_thr)].copy()
    rows = rows.sort_values("score", ascending=False).head(MAX_DETS_PER_IMAGE)

    image = Image.open(image_path).convert("RGB")
    scale = width / image.width
    main_h = int(round(image.height * scale))
    image_resized = image.resize((width, main_h), Image.Resampling.LANCZOS)
    draw = ImageDraw.Draw(image_resized)

    colors = ["#2563eb", "#dc2626", "#16a34a", "#f97316", "#9333ea", "#0891b2"]
    for idx, (_, row) in enumerate(rows.iterrows()):
        x, y, w, h = [float(row[c]) * scale for c in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]]
        color = colors[idx % len(colors)] if int(row.get("keep", 1)) == 1 else "#9ca3af"
        draw.rectangle((x, y, x + w, y + h), outline=color, width=4)
        label = f"{int(row['category_id']):05d} {row_drug_name(row)[:16]} {float(row.get('score', 0)):.2f}"
        tw = draw.textbbox((0, 0), label, font=FONT_SMALL)[2]
        draw.rectangle((x, max(0, y - 20), x + tw + 8, max(20, y)), fill=color)
        draw.text((x + 4, max(0, y - 18)), label, font=FONT_SMALL, fill="white")

    side_w = 380
    canvas = Image.new("RGB", (width + side_w, max(main_h, 170 + 124 * max(1, len(rows)))), "white")
    canvas.paste(image_resized, (0, 0))
    side = ImageDraw.Draw(canvas)
    sx = width + 14
    side.text((sx, 14), f"test image {image_id}", font=FONT_LABEL, fill="#111827")
    side.text((sx, 42), "candidate labels from submission", font=FONT_SMALL, fill="#6b7280")

    y0 = 72
    for idx, (_, row) in enumerate(rows.iterrows()):
        y = y0 + idx * 124
        crop = reference_crop(int(row["category_id"]), size=(96, 96))
        canvas.paste(crop, (sx, y))
        side.rectangle((sx, y, sx + 96, y + 96), outline="#d1d5db")
        tx = sx + 110
        side.text((tx, y), f"{int(row['category_id']):05d}", font=FONT_SMALL, fill="#2563eb")
        side.text((tx, y + 19), row_drug_name(row)[:22], font=FONT_SMALL, fill="#111827")
        side.text((tx, y + 38), str(row.get("name_en", ""))[:28], font=FONT_TINY, fill="#4b5563")
        side.text((tx, y + 57), f"score {float(row.get('score', 0)):.3f} · keep {row.get('keep', '')}", font=FONT_TINY, fill="#6b7280")
        side.text((tx, y + 74), f"rank {int(row.get('candidate_rank', idx + 1))}", font=FONT_TINY, fill="#6b7280")

    return canvas


def make_contact_sheet(page=0, count=18, cols=2, score_thr=0.0):
    review_df = pd.read_csv(PSEUDO_REVIEW_CSV)
    image_ids = sorted(review_df["image_id"].astype(int).unique())
    selected = image_ids[page * count:(page + 1) * count]
    if not selected:
        raise ValueError("No image ids for this page")

    tiles = [draw_test_review_image(image_id, review_df=review_df, score_thr=score_thr, width=520) for image_id in selected]
    gap = 18
    rows = math.ceil(len(tiles) / cols)
    tile_w = max(t.width for t in tiles)
    tile_h = max(t.height for t in tiles)
    canvas = Image.new("RGB", (cols * tile_w + (cols - 1) * gap, rows * tile_h + (rows - 1) * gap), "#f3f4f6")
    for i, tile in enumerate(tiles):
        x = (i % cols) * (tile_w + gap)
        y = (i // cols) * (tile_h + gap)
        canvas.paste(tile, (x, y))
    out = OUT_DIR / f"test_review_contact_sheet_page_{page:03d}.png"
    canvas.save(out, quality=95)
    return out

# First review page. Change page/count if you want more sheets.
contact_sheet = make_contact_sheet(page=0, count=12, cols=2, score_thr=SCORE_THR_SEED)
print(contact_sheet)
display(IPyImage(filename=str(contact_sheet)))


## 5. Build Pseudo-Ground-Truth Annotation Files

`test_pseudo_ground_truth_review.csv`를 검수한 뒤 아래 셀을 실행하면, 평가에 쓸 정답 annotation 파일을 만듭니다.

`correct_n_number`가 비어 있으면 `predicted_n_number`를 그대로 사용합니다. `correct_n_number`에 `N07` 같은 값이 있으면 그 번호가 최종 정답 class가 됩니다.

기본값은 `require_reviewed=False`입니다. 즉 아직 `review_status=reviewed`를 다 채우지 않아도 파일은 만들어집니다. 엄격하게 검수 완료 row만 쓰고 싶으면 `require_reviewed=True`로 바꾸세요.


In [ ]:
def parse_keep(value):
    if pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {"1", "true", "yes", "y", "keep"}
    return bool(int(value))


def image_size_for_test_id(image_id):
    path = TEST_IMG_DIR / f"{int(image_id)}.png"
    if not path.exists():
        raise FileNotFoundError(path)
    with Image.open(path) as img:
        return img.width, img.height


def resolve_review_row_category(row):
    correct_n = normalize_n_number(row.get("correct_n_number", ""))
    predicted_n = normalize_n_number(row.get("predicted_n_number", ""))
    resolved_n = correct_n or predicted_n
    if resolved_n:
        category_id = category_id_from_n_number(resolved_n)
        return resolved_n, int(category_id)
    return category_to_n_number.get(int(row["category_id"]), ""), int(row["category_id"])


def build_pseudo_ground_truth(require_reviewed=False):
    review = pd.read_csv(PSEUDO_REVIEW_CSV)
    review["keep_bool"] = review["keep"].map(parse_keep)
    kept = review[review["keep_bool"]].copy()

    if require_reviewed:
        kept = kept[kept["review_status"].fillna("").str.lower().eq("reviewed")].copy()

    if "predicted_n_number" not in kept.columns:
        kept["predicted_n_number"] = kept["category_id"].astype(int).map(category_to_n_number)
    if "correct_n_number" not in kept.columns:
        kept["correct_n_number"] = ""

    kept["predicted_category_id"] = kept["category_id"].astype(int)
    resolved = kept.apply(resolve_review_row_category, axis=1, result_type="expand")
    kept["resolved_n_number"] = resolved[0]
    kept["category_id"] = resolved[1].astype(int)

    unknown = sorted(set(kept["category_id"]) - set(ref["category_id"].astype(int)))
    if unknown:
        raise ValueError(f"Unknown category_id in reviewed pseudo GT: {unknown[:20]}")

    for col in ["bbox_x", "bbox_y", "bbox_w", "bbox_h"]:
        kept[col] = pd.to_numeric(kept[col], errors="raise")
    kept = kept[(kept["bbox_w"] > 0) & (kept["bbox_h"] > 0)].copy()
    kept = kept.sort_values(["image_id", "candidate_rank", "score"], ascending=[True, True, False]).reset_index(drop=True)
    kept["annotation_id"] = np.arange(1, len(kept) + 1)

    name_map = ref.set_index("category_id")["name"].to_dict()
    kept["drug_name"] = kept["category_id"].map(name_map)

    output_cols = [
        "annotation_id", "image_id", "predicted_n_number", "correct_n_number", "resolved_n_number",
        "predicted_category_id", "category_id", "drug_name",
        "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score", "source", "review_status", "review_note"
    ]
    kept[output_cols].to_csv(PSEUDO_GT_CSV, index=False, encoding="utf-8-sig")

    image_ids = sorted(kept["image_id"].astype(int).unique())
    images = []
    for image_id in image_ids:
        width, height = image_size_for_test_id(image_id)
        images.append({"id": int(image_id), "file_name": f"{int(image_id)}.png", "width": width, "height": height})

    categories = [
        {"id": int(row.category_id), "name": str(row.name), "supercategory": "pill"}
        for row in ref[["category_id", "name"]].itertuples(index=False)
    ]
    annotations = []
    for row in kept.itertuples(index=False):
        bbox = [float(row.bbox_x), float(row.bbox_y), float(row.bbox_w), float(row.bbox_h)]
        annotations.append({
            "id": int(row.annotation_id),
            "image_id": int(row.image_id),
            "category_id": int(row.category_id),
            "bbox": bbox,
            "area": float(row.bbox_w) * float(row.bbox_h),
            "iscrowd": 0,
        })

    coco = {"images": images, "annotations": annotations, "categories": categories}
    PSEUDO_GT_COCO_JSON.write_text(json.dumps(coco, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("pseudo GT rows:", len(kept))
    print("pseudo GT images:", len(images))
    print("CSV:", PSEUDO_GT_CSV)
    print("COCO JSON:", PSEUDO_GT_COCO_JSON)
    return kept, coco

pseudo_gt_df, pseudo_gt_coco = build_pseudo_ground_truth(require_reviewed=False)
display(pseudo_gt_df.head(12))


## 6. Evaluation Helpers

아래 함수는 COCO mAP와 IoU 0.75 기준 confusion matrix를 만듭니다. test는 pseudo-GT 기준, validation은 실제 val annotation 기준으로 계산합니다.


In [ ]:
def submission_csv_to_detection_json(pred_csv, out_json, image_ids=None, score_thr=0.0):
    pred = pd.read_csv(pred_csv)
    if image_ids is not None:
        image_ids = set(int(x) for x in image_ids)
        pred = pred[pred["image_id"].astype(int).isin(image_ids)].copy()
    pred = pred[pred["score"].fillna(0) >= score_thr].copy()
    detections = []
    for row in pred.itertuples(index=False):
        detections.append({
            "image_id": int(row.image_id),
            "category_id": int(row.category_id),
            "bbox": [float(row.bbox_x), float(row.bbox_y), float(row.bbox_w), float(row.bbox_h)],
            "score": float(row.score),
        })
    out_json.write_text(json.dumps(detections, ensure_ascii=False) + "\n", encoding="utf-8")
    return out_json, detections


def run_coco_map(gt_json, pred_json, out_prefix):
    if COCO is None or COCOeval is None:
        raise ImportError("pycocotools is required")

    coco_gt = COCO(str(gt_json))
    coco_dt = coco_gt.loadRes(str(pred_json))

    eval_full = COCOeval(coco_gt, coco_dt, "bbox")
    eval_full.evaluate(); eval_full.accumulate(); eval_full.summarize()

    eval_high = COCOeval(coco_gt, coco_dt, "bbox")
    eval_high.params.iouThrs = np.arange(0.75, 0.96, 0.05)
    eval_high.evaluate(); eval_high.accumulate(); eval_high.summarize()

    metrics = pd.DataFrame([
        {"metric": "AP@[0.50:0.95]", "value": float(eval_full.stats[0])},
        {"metric": "AP@0.50", "value": float(eval_full.stats[1])},
        {"metric": "AP@0.75", "value": float(eval_full.stats[2])},
        {"metric": "AP@[0.75:0.95]", "value": float(eval_high.stats[0])},
        {"metric": "AR@[0.50:0.95]|maxDets=1", "value": float(eval_full.stats[6])},
        {"metric": "AR@[0.50:0.95]|maxDets=10", "value": float(eval_full.stats[7])},
        {"metric": "AR@[0.50:0.95]|maxDets=100", "value": float(eval_full.stats[8])},
    ])
    metrics_path = OUT_DIR / f"{out_prefix}_map_summary.csv"
    metrics.to_csv(metrics_path, index=False, encoding="utf-8-sig")
    print("metrics:", metrics_path)
    display(metrics)
    return metrics, eval_full, eval_high


def xywh_iou(a, b):
    ax, ay, aw, ah = [float(x) for x in a]
    bx, by, bw, bh = [float(x) for x in b]
    ax2, ay2 = ax + aw, ay + ah
    bx2, by2 = bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def load_coco_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def class_name_map_from_coco(coco):
    return {int(c["id"]): str(c.get("name", c["id"])) for c in coco["categories"]}


def short_label(category_id, name_map):
    name = name_map.get(int(category_id), str(category_id))
    return f"{int(category_id):05d}\n{name[:22]}"


def detection_confusion(gt_json, pred_json, out_prefix, iou_thr=0.75, score_thr=0.0, max_dets=100):
    gt = load_coco_json(gt_json)
    pred = load_coco_json(pred_json)
    name_map = class_name_map_from_coco(gt)
    cat_ids = sorted(name_map)
    bg = -1
    matrix_ids = cat_ids + [bg]

    gt_by_img = defaultdict(list)
    for ann in gt["annotations"]:
        gt_by_img[int(ann["image_id"])].append(dict(ann))

    pred_by_img = defaultdict(list)
    for p in pred:
        if float(p.get("score", 0)) >= score_thr:
            pred_by_img[int(p["image_id"])].append(dict(p))
    for image_id in list(pred_by_img):
        pred_by_img[image_id] = sorted(pred_by_img[image_id], key=lambda x: float(x.get("score", 0)), reverse=True)[:max_dets]

    conf = pd.DataFrame(0, index=matrix_ids, columns=matrix_ids, dtype=int)
    matches = []

    for image_id in sorted(set(gt_by_img) | set(pred_by_img)):
        gts = gt_by_img.get(image_id, [])
        preds = pred_by_img.get(image_id, [])
        matched = set()
        for pred_rank, p in enumerate(preds, start=1):
            best_iou, best_idx, best_gt = 0.0, None, None
            for i, gt_ann in enumerate(gts):
                if i in matched:
                    continue
                iou = xywh_iou(p["bbox"], gt_ann["bbox"])
                if iou > best_iou:
                    best_iou, best_idx, best_gt = iou, i, gt_ann
            pred_cat = int(p["category_id"])
            if best_gt is not None and best_iou >= iou_thr:
                gt_cat = int(best_gt["category_id"])
                matched.add(best_idx)
                conf.loc[gt_cat, pred_cat] += 1
                match_type = "TP" if gt_cat == pred_cat else "CLASS_ERROR"
                matches.append({
                    "image_id": image_id, "pred_rank": pred_rank, "match_type": match_type,
                    "iou": best_iou, "score": float(p.get("score", 0)),
                    "gt_category_id": gt_cat, "gt_name": name_map.get(gt_cat, ""),
                    "pred_category_id": pred_cat, "pred_name": name_map.get(pred_cat, ""),
                    "gt_bbox": best_gt["bbox"], "pred_bbox": p["bbox"],
                })
            else:
                conf.loc[bg, pred_cat] += 1
                matches.append({
                    "image_id": image_id, "pred_rank": pred_rank, "match_type": "FP",
                    "iou": best_iou, "score": float(p.get("score", 0)),
                    "gt_category_id": "", "gt_name": "",
                    "pred_category_id": pred_cat, "pred_name": name_map.get(pred_cat, ""),
                    "gt_bbox": "", "pred_bbox": p["bbox"],
                })
        for i, gt_ann in enumerate(gts):
            if i in matched:
                continue
            gt_cat = int(gt_ann["category_id"])
            conf.loc[gt_cat, bg] += 1
            matches.append({
                "image_id": image_id, "pred_rank": "", "match_type": "FN",
                "iou": "", "score": "", "gt_category_id": gt_cat, "gt_name": name_map.get(gt_cat, ""),
                "pred_category_id": "", "pred_name": "", "gt_bbox": gt_ann["bbox"], "pred_bbox": "",
            })

    labels = [short_label(cid, name_map) if cid != bg else "__background__" for cid in matrix_ids]
    conf_named = conf.copy()
    conf_named.index = labels
    conf_named.columns = labels
    conf_csv = OUT_DIR / f"{out_prefix}_confusion_matrix_iou{int(iou_thr*100)}.csv"
    conf_named.to_csv(conf_csv, encoding="utf-8-sig")

    matches_df = pd.DataFrame(matches)
    matches_csv = OUT_DIR / f"{out_prefix}_matches_iou{int(iou_thr*100)}.csv"
    matches_df.to_csv(matches_csv, index=False, encoding="utf-8-sig")

    active = [cid for cid in matrix_ids if conf.loc[cid].sum() > 0 or conf[cid].sum() > 0]
    active_conf = conf.loc[active, active]
    active_labels = [short_label(cid, name_map) if cid != bg else "__background__" for cid in active]
    annot = active_conf.astype(str).mask(active_conf == 0, "")

    plt.figure(figsize=(max(12, len(active) * 0.52), max(10, len(active) * 0.48)))
    if sns is not None:
        sns.heatmap(active_conf, cmap="Blues", annot=annot, fmt="", cbar=True, linewidths=0.3, linecolor="#e5e7eb")
    else:
        plt.imshow(active_conf.values, cmap="Blues")
        plt.colorbar()
    plt.xticks(np.arange(len(active)) + 0.5, active_labels, rotation=90, fontsize=7)
    plt.yticks(np.arange(len(active)) + 0.5, active_labels, rotation=0, fontsize=7)
    plt.xlabel("Predicted class")
    plt.ylabel("Ground-truth class")
    plt.title(f"{out_prefix} Confusion Matrix @ IoU {iou_thr:.2f}")
    plt.tight_layout()
    conf_png = OUT_DIR / f"{out_prefix}_confusion_matrix_iou{int(iou_thr*100)}.png"
    plt.savefig(conf_png, dpi=220)
    plt.close()

    tp = int(sum(conf.loc[c, c] for c in cat_ids))
    fp = int(conf.loc[bg, cat_ids].sum() + sum(conf.loc[g, p] for g in cat_ids for p in cat_ids if g != p))
    fn = int(conf.loc[cat_ids, bg].sum() + sum(conf.loc[g, p] for g in cat_ids for p in cat_ids if g != p))
    summary = {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "micro_precision": tp / (tp + fp) if tp + fp else np.nan,
        "micro_recall": tp / (tp + fn) if tp + fn else np.nan,
        "confusion_csv": str(conf_csv),
        "confusion_png": str(conf_png),
        "matches_csv": str(matches_csv),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    display(IPyImage(filename=str(conf_png)))
    return conf_named, matches_df, summary


## 7. Evaluate Test Submission Against Pseudo-GT

이 점수는 **검수한 pseudo-GT 기준 점수**입니다. `test_pseudo_ground_truth_review.csv`를 아직 사람이 고치지 않았다면 사실상 모델이 자기 예측을 정답 seed로 삼은 것이므로 의미가 약합니다.


In [ ]:
test_image_ids = [img["id"] for img in pseudo_gt_coco["images"]]
TEST_DET_JSON = OUT_DIR / "submission_detections_for_pseudo_gt.json"
submission_csv_to_detection_json(SUBMISSION_CSV, TEST_DET_JSON, image_ids=test_image_ids, score_thr=SCORE_THR_SEED)

print("Pseudo-GT evaluation uses:")
print("GT:", PSEUDO_GT_COCO_JSON)
print("Pred:", TEST_DET_JSON)

test_metrics, test_eval_full, test_eval_high = run_coco_map(PSEUDO_GT_COCO_JSON, TEST_DET_JSON, "test_pseudo")
test_conf, test_matches, test_conf_summary = detection_confusion(
    PSEUDO_GT_COCO_JSON, TEST_DET_JSON, "test_pseudo", iou_thr=IOU_THRESHOLD, score_thr=SCORE_THR_SEED, max_dets=MAX_DETS_PER_IMAGE
)


## 8. Validation Evaluation With Real Val Annotations

이건 숨겨진 test가 아니라 우리가 만든 validation split의 실제 정답 annotation으로 계산합니다. `idx2cat.json`을 이용해 내부 class index를 원래 `category_id`로 되돌린 뒤, 같은 리포트 형식으로 mAP/confusion matrix를 냅니다.


In [ ]:
def remap_val_to_original_category_ids():
    idx2cat = {int(k): int(v) for k, v in json.loads(IDX2CAT_JSON.read_text()).items()}
    name_by_orig = ref.set_index("category_id")["name"].to_dict()

    gt = json.loads(VAL_GT_JSON.read_text())
    pred = json.loads(VAL_PREDS_JSON.read_text())

    out_gt = json.loads(json.dumps(gt))
    out_gt["categories"] = [
        {"id": int(orig), "name": str(name_by_orig.get(int(orig), orig)), "supercategory": "pill"}
        for _, orig in sorted(idx2cat.items())
    ]
    for ann in out_gt["annotations"]:
        ann["category_id"] = idx2cat[int(ann["category_id"])]

    out_pred = []
    for p in pred:
        row = dict(p)
        row["category_id"] = idx2cat[int(row["category_id"])]
        out_pred.append(row)

    gt_path = OUT_DIR / "val_ground_truth_original_category_ids.json"
    pred_path = OUT_DIR / "val_predictions_original_category_ids.json"
    gt_path.write_text(json.dumps(out_gt, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    pred_path.write_text(json.dumps(out_pred, ensure_ascii=False) + "\n", encoding="utf-8")
    return gt_path, pred_path

VAL_GT_ORIG_JSON, VAL_PREDS_ORIG_JSON = remap_val_to_original_category_ids()
val_metrics, val_eval_full, val_eval_high = run_coco_map(VAL_GT_ORIG_JSON, VAL_PREDS_ORIG_JSON, "val_real")
val_conf, val_matches, val_conf_summary = detection_confusion(
    VAL_GT_ORIG_JSON, VAL_PREDS_ORIG_JSON, "val_real", iou_thr=IOU_THRESHOLD, score_thr=SCORE_THR_SEED, max_dets=100
)


## 9. Output Files

핵심 파일은 아래 폴더에 저장됩니다.


In [ ]:
for path in sorted(OUT_DIR.glob("*")):
    print(path.name, f"{path.stat().st_size / 1024:.1f} KiB")
